# KrishiDisha tabular models - training on Kaggle

Runs `ml.train_tabular` (crop recommendation, fertilizer on the 750k-row Playground S5E6 table, yield with the time-split protocol) on a T4 with GPU XGBoost. Outputs land in `/kaggle/working/models` (private kernel: the .pkl weights are never published; copy `metrics_tabular.json` and `reports/` back into the repo).


In [ ]:
# Source comes from the private Kaggle dataset (git archive of the lab repo), not from GitHub.
# Kaggle extracts uploaded zips, so the tree is at .../krishidisha-lab-src/src/krishidisha; fall back to src.zip.
import glob, os, shutil, zipfile
shutil.rmtree("/tmp/krishidisha", ignore_errors=True)
_dirs = glob.glob("/kaggle/input/**/krishidisha-lab-src/src/krishidisha", recursive=True)
if _dirs:
    shutil.copytree(_dirs[0], "/tmp/krishidisha")
    _ver = os.path.join(os.path.dirname(os.path.dirname(_dirs[0])), "VERSION.txt")
else:
    _zip = glob.glob("/kaggle/input/**/krishidisha-lab-src/**/src.zip", recursive=True)[0]
    zipfile.ZipFile(_zip).extractall("/tmp")
    _ver = os.path.join(os.path.dirname(_zip), "VERSION.txt")
print("source version:", open(_ver).read().strip() if os.path.exists(_ver) else "unknown", "->", os.listdir("/tmp/krishidisha")[:8])
TASKS = "fertilizer"          # e.g. "crop fertilizer yield"
import os, subprocess, glob
W = "/kaggle/working"
os.chdir("/tmp/krishidisha")
# CPU only: tabular training never uses a GPU (owner rule)
FERT = (glob.glob("/kaggle/input/**/playground-series-s5e6/train.csv", recursive=True) or glob.glob("/kaggle/input/**/train.csv", recursive=True))[0]
print(FERT)


In [ ]:
# the trainer imports krishidisha.services.ml, which pulls in the Flask package: install the light web deps too
CMD = "pip install -q optuna flask flask-sqlalchemy python-dotenv 2>&1 | tail -1"
!{CMD}
CMD = f"nproc; free -g | head -2; python -u -m ml.train_tabular --data-dir data --output {W}/models --tasks {TASKS} --fert-data {FERT} 2>&1 | tee {W}/tabular.log | grep -v Warning | tail -40"
!{CMD}


In [ ]:
import json, shutil
m = json.load(open(f"{W}/models/metrics_tabular.json"))
print(json.dumps({k: v for k, v in m.get("fertilizer", m).items() if not isinstance(v, (list, dict))}, indent=1)[:3000])
shutil.rmtree("/tmp/krishidisha/.git", ignore_errors=True)
